In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import GDLModules.util as util
import plotly.express as px
import torch.nn as nn
import torch
import GDLModules.A_CAE as A_CAE
import GDLModules.A_AE as A_AE
from datasets import load_dataset
import matplotlib.pyplot as plt
from sklearn.datasets import make_s_curve

import copy
import panel as pn

pn.extension("plotly", comms="ipywidgets")

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
torus_data = util.generate_torus(10000)
torus_data_test = util.generate_torus(10000)
util.plot3d([(torus_data, "torus data")], title="Double Torus")

In [12]:
X_train = torch.tensor(torus_data, dtype=torch.float32).to(device)
X_test = torch.tensor(torus_data_test, dtype=torch.float32).to(device)
X_loader_train = torch.utils.data.DataLoader(X_train, batch_size=1024, shuffle=True)
X_loader_test = torch.utils.data.DataLoader(X_test, batch_size=1024, shuffle=True)

In [37]:
cae = A_CAE.CAE(x_dim=3, z_dim=2, hidden_structure=[128, 128], chart_amount=6, activation=nn.Tanh()).to(device)

A_CAE.pre_train_cae(num_epochs=200, x_train=X_train, network=cae, device = device, lr=1e-2)
pretrained = copy.deepcopy(cae)


100%|██████████| 200/200 [00:04<00:00, 43.58it/s]


In [49]:

A_CAE.reverse_train_cae(num_epochs=100, x_train=X_train, network=cae, device = device, lr=1e-3, batch_size=5120, lipschitz_weight=1.00, spawn_interval=999, criterion_type="chamfer")



  8%|▊         | 8/100 [00:45<08:47,  5.74s/it, chart_amount=13, loss=0.0143]


KeyboardInterrupt: 

In [50]:
lst = []
colors = []
for chart_idx in range(cae.chart_amount):
    z_samples = cae.sample_z(n=1000)
    x_decoded=cae.decode(z_samples, chart_idx)
    lst.append(x_decoded.detach().cpu().numpy())
    colors += [chart_idx]* len(x_decoded)
    
points = np.concatenate(lst, axis=0)
util.plot3d([(points, "decoded points")], title="Decoded Points from Post-trained CAE", color=[np.array(colors)])
    

In [44]:
np.array(colors)

array([ 0,  0,  0, ..., 16, 16, 16], shape=(6800,))

In [23]:


from GDLModules.viser_viz import open_split_view


sw = open_split_view(X_test, cae)
print("Press Enter to close the viewer...")
input()
sw.close()


╭──────────────── viser ────────────────╮
│             ╷                         │
│   HTTP      │ http://127.0.0.1:8080   │
│   Websocket │ ws://127.0.0.1:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

╭──────────────── viser ────────────────╮
│             ╷                         │
│   HTTP      │ http://127.0.0.1:8081   │
│   Websocket │ ws://127.0.0.1:8081     │
│             ╵                         │
╰───────────────────────────────────────╯

Press Enter to close the viewer...


(viser) Connection opened (0, 1 total), 77 persistent messages

(viser) Connection opened (0, 1 total), 95 persistent messages

(viser) Connection closed (0, 0 total)

(viser) Connection closed (0, 0 total)

(viser) Server stopped

(viser) Server stopped

# Visual training


In [8]:
cae = A_CAE.CAE(x_dim=3, z_dim=2, hidden_structure=[15,15], chart_amount=10, activation=nn.Tanh()).to(device)

A_CAE.pre_train_cae(num_epochs=50, x_train=X_train, network=cae, device = device, lr=1e-2)
pretrained = copy.deepcopy(cae)


100%|██████████| 50/50 [00:02<00:00, 22.44it/s]


In [10]:

import copy
import threading
import time

import torch
import plotly.graph_objects as go
from IPython.display import display

# --------------------------------------------------
# Model and evaluation data
# --------------------------------------------------
restart_training = False

if(restart_training):
    cae = copy.deepcopy(pretrained)

plot_data = X_test[:10000].detach()
plot_data = plot_data.to(next(cae.parameters()).device)

stop_event = threading.Event()


threshold = 0.005

@torch.no_grad()
def plot_values():
    points = []
    colors = []

    for i in range(cae.chart_amount):
        reconstructed = cae.decode(cae.encode(plot_data, i), i)
        errors = ((reconstructed - plot_data) ** 2).sum(dim=1)
        accepted = errors < threshold

        selected = reconstructed[accepted]
        points.append(selected)
        colors.append(torch.full(
            (len(selected),),
            i,
            device=plot_data.device,
            dtype=torch.long,
        ))

    return (
        torch.cat(points, dim=0).cpu().numpy(),
        torch.cat(colors, dim=0).cpu().numpy(),
    )

# --------------------------------------------------
# Record history
# --------------------------------------------------

history = []
start_time = time.monotonic()


def update():
    points, colors = plot_values()
    elapsed = time.monotonic() - start_time

    history.append((
        elapsed,
        points.copy(),
        colors.copy(),
    ))


# --------------------------------------------------
# Start training
# --------------------------------------------------

thread = threading.Thread(
    target=A_CAE.train_cae,
    args=(200, X_loader_train, cae, device),
    kwargs={
        "lr": 1e-3,
        "best_k": 1,
        "mmd_weight": 0.0,
        "lipschitz_weight": 0.0,
        "stop_event": stop_event,
        "threshold": threshold,
    },
)

thread.start()


# --------------------------------------------------
# Original loop
# --------------------------------------------------

try:
    while True:
        update()
        time.sleep(1)

except KeyboardInterrupt:
    print("Stopped")
    stop_event.set()


# --------------------------------------------------
# Construct interactive history
# --------------------------------------------------

def make_history_plot(history):
    if not history:
        raise ValueError("No snapshots were recorded.")

    t0, points, colors = history[0]

    fig = util.plot3d(
        [(points, "reconstructed")],
        title="Training reconstruction",
        color=[colors],
    )

    fig.update_layout(
        uirevision="keep-camera",
        showlegend=False,
    )

    fig.update_scenes(
        uirevision="keep-camera",
        aspectmode="cube",
        xaxis_range=[-1.5, 1.5],
        yaxis_range=[-1.5, 1.5],
        zaxis_range=[-1.5, 1.5],
    )

    # Each frame contains a complete saved reconstruction
    frames = []

    for i, (elapsed, points, colors) in enumerate(history):
        frames.append(
            go.Frame(
                name=str(i),
                data=[
                    go.Scatter3d(
                        x=points[:, 0],
                        y=points[:, 1],
                        z=points[:, 2],
                        marker=dict(color=colors),
                    )
                ],
                traces=[0],
            )
        )

    fig.frames = frames

    # Plotly-native slider
    steps = []

    for i, (elapsed, _, _) in enumerate(history):
        steps.append(
            dict(
                method="animate",
                label=f"{elapsed:.1f}",
                args=[
                    [str(i)],
                    dict(
                        mode="immediate",
                        frame=dict(
                            duration=0,
                            redraw=True,
                        ),
                        transition=dict(duration=0),
                    ),
                ],
            )
        )

    fig.update_layout(
        sliders=[
            dict(
                active=0,
                currentvalue=dict(
                    prefix="Training time: ",
                    suffix=" s",
                ),
                pad=dict(t=40),
                steps=steps,
            )
        ],
        margin=dict(b=100),
    )

    return fig

import plotly.io as pio

pio.renderers.default = "browser"
# Show the saved history after stopping
fig = make_history_plot(history)
fig.show()


100%|██████████| 200/200 [07:04<00:00,  2.12s/it, loss=-0.5719%, lr=0.001000, mmd_loss=12.2356%, rec_loss=-0.5631%]


Stopped


In [16]:
stop_event.set()
server.stop()

 33%|███▎      | 66/200 [01:03<01:50,  1.22it/s, loss=0.2673%, lr=0.001000, rec_loss=0.2718%]

 34%|███▎      | 67/200 [01:03<02:06,  1.05it/s, loss=0.2658%, lr=0.001000, rec_loss=0.2551%]


Training stopped.


# Graph experiments


In [7]:
if "graph_server" in globals():
    graph_server.stop()

import importlib
import GDLModules.graph_viz as graph_viz

importlib.reload(graph_viz)

graph_server = graph_viz.open_graph_view(
    X_train, radius=0.05, k=10, port=5006
)

Launching server at http://127.0.0.1:5006
